# 04  Bronze to Silver

## Purpose

This notebook develops the Bronze → Silver transformation for the OpenAQ warehouse.

Bronze contains schema-controlled, source-aligned datasets derived from immutable Raw data. Silver converts those datasets into clean, deduplicated, relationship-aware analytical data.

Development sequence:

1. Read Bronze Delta datasets.
2. Validate the Bronze inputs required by Silver.
3. Resolve repeated metadata snapshots.
4. Validate entity relationships.
5. Clean and deduplicate measurements.
6. Apply analytical data-quality rules.
7. Build Silver datasets.
8. Persist them as Delta.
9. Re-read and reconcile Silver against Bronze.

Source lineage is retained so transformed records remain traceable to their Bronze and Raw origins.
**Packaging note:** the transformation logic developed in this notebook now lives in `spark/transformations/silver.py` (`build_silver`). This notebook is a thin runner that imports the packaged module and executes it against the configured catalog.


## Configure the Silver run

Silver reads only persisted Bronze Delta tables from Unity Catalog and persists managed Silver tables. The packaged `spark/` transformations are imported from the bundle-deployed package root.


In [ ]:
import sys

PACKAGE_ROOT = "/Workspace/Users/dammifabz@gmail.com/.bundle/openaq-databricks/dev/files"  # bundle-deployed spark/ package; adjust for exploration
if PACKAGE_ROOT not in sys.path:
    sys.path.append(PACKAGE_ROOT)

from spark.transformations.silver import build_silver

CATALOG = "workspace"


## Run the Bronze to Silver transformation

`build_silver` loads the persisted Bronze datasets, validates the input contract, resolves metadata snapshots, enforces relationship and measurement-grain checks, separates analytical measurements from flagged null observations, persists the six Silver managed Delta tables, and returns the verified persisted row counts.


In [ ]:
counts = build_silver(spark, CATALOG)

for table_name, row_count in counts.items():
    print(f"{table_name}: {row_count:,} rows persisted")
